# Phase 5 — Baseline & Ensemble Models
Competition: Playground Series S6E7 — Predicting Student Health Risk  
Metric: **Balanced accuracy** (average per-class recall)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from src.data_prep import load_data, split_data, get_column_types
from src.features import fit_and_save_preprocessor
from src.train import train_logistic, train_random_forest, cross_validate, save_model
from src.evaluate import evaluate_sklearn, plot_confusion_matrix

## 1. Load & split data

In [ ]:
train_df, test_df = load_data('../data/raw')
X_train, X_val, y_train, y_val = split_data(train_df)

print('Train size:', X_train.shape)
print('Val size:  ', X_val.shape)

print('\nClass balance (train):')
print(y_train.value_counts(normalize=True).round(3))

## 2. Build & fit preprocessing pipeline

In [ ]:
numeric_cols, categorical_cols = get_column_types(X_train)
print('Numeric cols:    ', numeric_cols)
print('Categorical cols:', categorical_cols)

preprocessor = fit_and_save_preprocessor(
    X_train, numeric_cols, categorical_cols,
    save_path='../models/preprocessor.pkl'
)

X_train_proc = preprocessor.transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
print('Processed shape:', X_train_proc.shape)

## 3. Logistic Regression — baseline

In [ ]:
lr = train_logistic(X_train_proc, y_train)
lr_bal_acc, lr_preds = evaluate_sklearn(lr, X_val_proc, y_val, 'Logistic Regression')
plot_confusion_matrix(y_val, lr_preds, lr.classes_, title='LR Confusion Matrix')

## 4. Random Forest with GridSearchCV

In [ ]:
best_rf = train_random_forest(X_train_proc, y_train, cv=5)
rf_bal_acc, rf_preds = evaluate_sklearn(best_rf, X_val_proc, y_val, 'Random Forest (tuned)')
plot_confusion_matrix(y_val, rf_preds, best_rf.classes_, title='RF Confusion Matrix')

## 5. Cross-validate best candidate

In [ ]:
cv_scores = cross_validate(best_rf, X_train_proc, y_train, cv=5)
print(f'RF CV balanced accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 6. Results summary

In [ ]:
results = pd.DataFrame([
    {'Model': 'LogisticRegression (balanced)', 'Val balanced_acc': lr_bal_acc, 'CV mean': '-', 'CV std': '-', 'Notes': 'baseline'},
    {'Model': 'RandomForest (tuned, balanced)', 'Val balanced_acc': rf_bal_acc, 'CV mean': round(cv_scores.mean(),4), 'CV std': round(cv_scores.std(),4), 'Notes': str(best_rf.get_params())},
])
print(results.to_markdown(index=False))
results.to_csv('../models/results_so_far.csv', index=False)

## 7. Save best sklearn model so far

In [ ]:
# Save whichever is better
best_model = best_rf if rf_bal_acc >= lr_bal_acc else lr
save_model(best_model, '../models/model_final.pkl')
print('Saved model type:', type(best_model).__name__)